## Round Robin Scheduling Algorithm Demo

Demo for Round Robin scheduling algorithm with 8 processes


In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))  # add parent directory to search path

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from core.process import Process
from core.scheduler_rr import round_robin
from core.process_generator import generate_processes
import plotly.figure_factory as ff
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display


## Create Sample Processes

creating 8 processes with different burst times and arrival times to run the basic Round Robin scheduler.

In [3]:
# Create 8 processes with varying burst times and arrival times
datasets = {
    "20_processes": generate_processes(20, seed=1),
    "50_processes": generate_processes(50, seed=2),
    "80_processes": generate_processes(80, seed=3),
    "100_processes": generate_processes(100, seed=4),
    "1000_processes": generate_processes(1000, seed=5)
}

def show_processes(processes):
    df = pd.DataFrame([{
        'PID': p.pid,
        'Arrival Time': p.arrival_time,
        'Burst Time': p.burst_time
    } for p in processes])
    display(df.head(10))
    print(f"Total processes: {len(processes)}")

show_processes(datasets["1000_processes"])


,PID,Arrival Time,Burst Time
0,1,2,13
1,2,6,18
2,3,6,16
3,4,10,9
4,5,13,7
5,6,13,17
6,7,19,14
7,8,21,20
8,9,21,8
9,10,22,7


Total processes: 1000


## Run the Round Robin algorithm

Running the round robin algorithm with the processes and time quantum of 3

In [4]:
context_switch_time = 1
processes = datasets["1000_processes"]


def build_results_df(metrics):
    return pd.DataFrame([{
        'Process ID': p.pid,
        'Arrival Time': p.arrival_time,
        'Burst Time': p.burst_time,
        'Completion Time': p.completion_time,
        'Turnaround Time': p.turnaround_time,
        'Waiting Time': p.waiting_time,
        'Start Time': p.start_time,
        'First Response Time': metrics["first_response_times"].get(p.pid, None),
    } for p in metrics['completed_processes']
    
    ])



time_quantum = 3
metrics = round_robin(processes, time_quantum, context_switch_time)
results_df = build_results_df(metrics)

## Metrics

Visualizing the waiting times and turnaround times for the processes.

In [6]:
# Create bar chart comparing waiting times and turnaround times
fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['Process ID'],
    y=results_df['Waiting Time'],
    name='Waiting Time',
    marker_color='rgb(55, 83, 109)'
))

fig.add_trace(go.Bar(
    x=results_df['Process ID'],
    y=results_df['Turnaround Time'],
    name='Turnaround Time',
    marker_color='rgb(26, 118, 255)'
))

fig.update_layout(
    title='Process Waiting and Turnaround Times',
    xaxis_title='Process ID',
    yaxis_title='Time Units',
    barmode='group',
    bargap=0.15,
    bargroupgap=0.1
)

fig.show()

# Print average metrics
print(f"Average Turnaround Time: {metrics['average_turnaround_time']:.2f}")
print(f"Average Waiting Time: {metrics['average_waiting_time']:.2f}")
print(f"CPU Utilization: {metrics['cpu_utilization']:.2f}%")
print(f"Context Switches: {metrics['context_switches']}")
print(f"Total Time: {metrics['execution_time']}")
print(f"Average First Response Time: {metrics['average_first_response_time']:.2f}")


if 'First Response Time' in results_df.columns:
    fig3 = go.Figure()
    fig3.add_trace(go.Bar(
        x=results_df['Process ID'],
        y=results_df['First Response Time'],
        name='First Response Time',
        marker_color='rgb(255, 153, 51)'
    ))
    fig3.update_layout(
        title='Process First Response Time',
        xaxis_title='Process ID',
        yaxis_title='Time Units',
        barmode='group'
    )
    fig3.show()

# fig2 = go.Figure()
# fig2.add_trace(go.Scatter(
#     x=rq_df["Time"],
#     y=rq_df["RQ Length"],
#     mode='lines+markers',
#     name='Ready Queue Length',
#     line=dict(color='orange')
# ))
# fig2.update_layout(
#     title='Ready Queue Length Over Time',
#     xaxis_title='Time Units',
#     yaxis_title='Queue Length'
# )
# fig2.show()

Average Turnaround Time: 8660.35
Average Waiting Time: 8649.22
CPU Utilization: 99.99%
Context Switches: 4049
Total Time: 15187
Average First Response Time: 1763.66


## CPU Utilization

visualizing the CPU utilization and context switch overhead.

In [11]:
# Create pie chart for CPU utilization
labels = ['CPU Utilization', 'Context Switch Overhead', 'Idle Time']
values = [
    metrics['cpu_utilization'],
    metrics['context_switch_overhead'],
    100 - metrics['cpu_utilization'] - metrics['context_switch_overhead']
]

fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.3)])
fig.update_layout(title='CPU Time Distribution')
fig.show()